# LangGraph와 AgentCore 메모리 도구 (단기 메모리)

## 소개
이 노트북은 LangGraph 프레임워크를 사용하여 대화형 AI 에이전트에 Amazon Bedrock AgentCore 메모리 기능을 통합하는 방법을 보여줍니다. 단일 대화 세션 내에서 **단기 메모리** 유지에 초점을 맞추며, 명시적인 컨텍스트 관리 없이도 에이전트가 대화 초반의 정보를 기억할 수 있도록 합니다.


## 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형                                                                      |
| 에이전트 활용 사례  | 개인 피트니스                                                                    |
| 에이전트 프레임워크 | Langgraph                                                                        |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, Langgraph, 도구를 통한 메모리 조회                        |
| 예제 난이도         | 초급                                                                             |

학습 내용:
- AgentCore 메모리를 사용하여 단기 메모리용 메모리 저장소 생성
- LangGraph를 사용하여 구조화된 메모리 워크플로우가 있는 에이전트 생성
- 대화 기록 조회를 위한 메모리 도구 구현
- 단일 세션 내에서 컨텍스트 정보 접근 및 활용
- 효과적인 메모리 회상을 통한 대화 경험 향상


### 시나리오 컨텍스트

이 예제에서는 대화 중 언급되는 운동 세부 사항, 피트니스 목표, 신체적 제한 사항, 운동 선호도를 기억할 수 있는 "**개인 피트니스 코치**"를 만들겠습니다. 이 어시스턴트는 효과적인 단기 메모리 관리가 사용자에게 반복적인 정보 입력 없이도 더 자연스럽고 개인화된 피트니스 코칭 경험을 제공하는 방법을 보여줍니다.


## 아키텍처
<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리에 대한 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델 접근 권한

환경 설정부터 시작하겠습니다!

## 1단계: 환경 설정
이 노트북을 실행하는 데 필요한 모든 라이브러리를 임포트하고 클라이언트를 정의하는 것부터 시작하겠습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime

Amazon Bedrock 모델 및 AgentCore에 대한 적절한 권한이 있는 리전과 역할을 정의합니다.

In [ ]:
import os
region = os.getenv('AWS_REGION', 'us-west-2')

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

### 통합 작동 방식

LangGraph와 AgentCore 메모리 간의 통합은 다음을 포함합니다:

1. AgentCore 메모리를 사용하여 단기 메모리에 대화 저장
2. 메모리 작업을 관리하기 위한 LangGraph의 구조화된 워크플로우

이 접근 방식은 메모리 관리를 추론 과정과 분리하여 더 깔끔하고 유지보수하기 쉬운 에이전트 아키텍처를 만듭니다.

## 2단계: 메모리 생성
이 섹션에서는 AgentCore 메모리 SDK를 사용하여 메모리 저장소를 생성합니다. 이 메모리 저장소를 통해 에이전트가 대화에서의 정보를 유지할 수 있습니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from botocore.exceptions import ClientError

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "FitnessCoach"
memory_id = None

In [ ]:
try:
    print("메모리 생성 중...")
    # 메모리 리소스 생성
    memory = client.create_memory_and_wait(
        name=memory_name,                       # 이 이름은 이 계정의 모든 메모리에서 고유합니다
        description="Fitness Coach Agent",      # 사람이 읽을 수 있는 설명
        strategies=[],                          # 단기 메모리에는 메모리 전략 없음
        event_expiry_days=7,                    # 메모리는 7일 후 만료
        max_wait=300,                           # 메모리 생성 대기 최대 시간 (5분)
        poll_interval=10                        # 10초마다 상태 확인
    )

    # 메모리 ID 추출 및 출력
    memory_id = memory['id']
    logger.info(f"메모리가 성공적으로 생성되었습니다. ID: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하면 해당 ID를 가져옵니다
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 처리
    logger.info(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"메모리 정리 실패: {cleanup_error}")

## 3단계: LangGraph 에이전트 생성
LangGraph로 에이전트를 생성하는 데 필요한 모든 라이브러리를 임포트하겠습니다.

In [ ]:
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_aws import ChatBedrock

### LangGraph 에이전트 구현

이제 메모리 도구를 통합하여 LangGraph로 에이전트를 생성하겠습니다:

In [ ]:
def create_agent(client, memory_id, actor_id, session_id):
    """LangGraph 에이전트를 생성하고 구성"""
    
    # LLM 초기화 (필요에 따라 모델 및 파라미터 조정)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 또는 선호하는 모델
        model_kwargs={"temperature": 0.1}
    )
    
    @tool
    def list_events():
        """Tool used when needed to retrieve recent information""" 
        events = client.list_events(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                max_results=10
            )
        return events
        
    
    # LLM에 도구 바인딩
    tools = [list_events]
    llm_with_tools = llm.bind_tools(tools)
    
    # 시스템 메시지: 개인 피트니스 코치 - 운동 루틴 개발, 사용자 선호도/제한사항 기억, 맞춤형 피트니스 추천 제공
    system_message = """You are the Personal Fitness Coach, a sophisticated fitness guidance assistant.
                        PURPOSE:
                        - Help users develop workout routines based on their fitness goals
                        - Remember user's exercise preferences, limitations, and progress
                        - Provide personalized fitness recommendations and training plans
                        MEMORY CAPABILITIES:
                        - You have access to recent events with the list_events tool
                        """
    
    # 챗봇 노드 정의
    def chatbot(state: MessagesState):
        raw_messages = state["messages"]
    
        # 중복이나 잘못된 배치를 방지하기 위해 기존 시스템 메시지 제거
        non_system_messages = [msg for msg in raw_messages if not isinstance(msg, SystemMessage)]
    
        # SystemMessage를 항상 첫 번째로 배치
        messages = [SystemMessage(content=system_message)] + non_system_messages
    
        latest_user_message = next((msg.content for msg in reversed(messages) if isinstance(msg, HumanMessage)), None)
    
        # 도구가 바인딩된 모델에서 응답 받기
        response = llm_with_tools.invoke(messages)
    
        # 해당되는 경우 대화 저장
        if latest_user_message and response.content.strip():  # 응답에 내용이 있는지 확인
            conversation = [
                (latest_user_message, "USER"),
                (response.content, "ASSISTANT")
            ]
            
            # 모든 메시지 텍스트가 비어있지 않은지 검증
            if all(msg[0].strip() for msg in conversation):  # 빈 메시지가 없는지 확인
                try:
                    client.create_event(
                        memory_id=memory_id,
                        actor_id=actor_id,
                        session_id=session_id,
                        messages=conversation
                    )
                except Exception as e:
                    print(f"대화 저장 오류: {str(e)}")
        
        # 전체 메시지 기록에 응답 추가
        return {"messages": raw_messages + [response]}
    
    # 그래프 생성
    graph_builder = StateGraph(MessagesState)
    
    # 노드 추가
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))
    
    # 엣지 추가
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    
    # 진입점 설정
    graph_builder.set_entry_point("chatbot")
    
    # 그래프 컴파일
    return graph_builder.compile()

### 에이전트 호출 래퍼 생성

에이전트를 호출하기 위한 간단한 래퍼를 만들어 보겠습니다:

In [ ]:
def langgraph_bedrock(payload, agent):
    """
    페이로드를 사용하여 에이전트를 호출
    """
    user_input = payload.get("prompt")
    
    # LangGraph에서 기대하는 형식으로 입력 생성
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # 최종 메시지 내용 추출
    return response["messages"][-1].content

## 4단계: LangGraph 에이전트 실행
이제 AgentCore 메모리가 통합된 에이전트를 실행할 수 있습니다.

In [ ]:
# 이 대화를 위한 고유 액터 및 세션 ID 생성
actor_id = f"user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"workout-{datetime.now().strftime('%Y%m%d%H%M%S')}"

In [ ]:
# AgentCore 메모리가 통합된 에이전트 생성
agent = create_agent(client, memory_id, actor_id, session_id)

#### 축하합니다! 에이전트가 준비되었습니다!!

### 에이전트를 테스트해 봅시다

에이전트와 상호작용하여 메모리 기능을 테스트해 보겠습니다:

In [ ]:
# 사용자: 안녕하세요! 첫 날인데, 운동 루틴이 필요합니다.
response = langgraph_bedrock({"prompt": "Hello! This is my first day, I need a workout routine."}, agent)
print(f"에이전트: {response}\n")

In [ ]:
# 사용자: 근육을 키우고 싶고, 이두근 루틴을 찾고 있습니다. 허리 통증이 좀 있습니다.
response = langgraph_bedrock({"prompt": "I want to build muscle, looking for a biceps routine. I have some lower back problems."}, agent)
print(f"에이전트: {response}\n")

In [ ]:
# 사용자: 반복 횟수와 함께 세 가지 운동을 알려줄 수 있나요?
response = langgraph_bedrock({"prompt": "Can you give me three exercises with number of reps?"}, agent)
print(f"에이전트: {response}\n")

### 메모리 지속성 테스트

AgentCore 메모리 통합의 강력함을 제대로 보여주기 위해, 새로운 에이전트 인스턴스를 생성하고 이전 대화를 기억할 수 있는지 확인해 보겠습니다:

In [ ]:
# 새 에이전트 인스턴스 생성 (새 세션 시뮬레이션)
new_agent = create_agent(client, memory_id, actor_id, session_id)

# 새 에이전트가 우리의 선호도를 기억하는지 테스트
# 사용자: 다시 안녕하세요! 마지막 운동 세션에 대해 알려줄 수 있나요?
response = langgraph_bedrock({
    "prompt": "Hello again! Can you remind me about my last workout session?"
}, new_agent)

print("새 에이전트 세션:\n")
print(f"에이전트: {response}")

## 요약

이 노트북에서 다음 내용을 시연했습니다:

1. AI 에이전트를 위한 AgentCore 메모리 리소스 생성 방법
2. 메모리 통합이 있는 LangGraph 워크플로우 구축
3. 대화 기록 조회를 위한 메모리 도구 구현
4. 필요시 메모리를 지능적으로 사용하는 에이전트 생성
5. 에이전트 인스턴스 간 메모리 지속성 테스트

이 통합은 구조화된 워크플로우(LangGraph)와 강력한 메모리 시스템(AgentCore 메모리)을 결합하여 더 지능적이고 컨텍스트를 인식하는 AI 에이전트를 만드는 힘을 보여줍니다.

여기서 시연한 접근 방식은 멀티 에이전트 시스템, 추출 전략을 포함한 장기 메모리, 대화 컨텍스트 기반의 특화된 메모리 조회 등 더 복잡한 활용 사례로 확장할 수 있습니다.

## 정리
이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다.

In [ ]:
#client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)